# Procesar Video en Colab → Tracking (F1)

Corre el pipeline de tracking (L2) en **Google Colab** (Python 3.11 + GPU T4/P100), donde
el paquete `inference` de Roboflow funciona y descarga los modelos solos con tu API key.

**Por qué Colab:** el paquete `inference` no soporta Python 3.13 (el de tu PC), y una
T4/P100 es varias veces más rápida que la GTX 1050 Ti local.

**Flujo de trabajo:**
1. **En tu PC:** bajaste el video (`descargar_video.ipynb`) y lo subís a Google Drive.
2. **Acá (Colab):** procesás → genera `positions.csv` + `ball_positions.csv` (OPTA 0-100).
3. **En tu PC:** copiás esos archivos a `data/temporadas/{año}/partidos/{match_key}/tracking/`
   y corrés `analisis_video.ipynb` para validar contra Sofascore y graficar.

**Prueba F1:** arrancar con `START_SECONDS` = el segundo del saque inicial (para saltar
la previa de la transmisión) y `TEST_SECONDS = 900` (15 min) — no el partido entero.

> **Antes de empezar:** Runtime → Change runtime type → **GPU** (T4 alcanza).

## 1. Instalar dependencias (~2-3 min)

In [ ]:
!pip install -q inference supervision "git+https://github.com/roboflow/sports.git"
print("Dependencias instaladas. Si pide reiniciar el runtime, hacelo y volvé a correr desde acá.")

## 2. Montar Drive y configurar

Subí el `.mp4` del partido a tu Google Drive (ej. una carpeta `analizar-partido/videos/`).
Después completá las rutas, tu API key y el `START_SECONDS` del saque inicial acá abajo.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CONFIGURAR ACÁ
ROBOFLOW_API_KEY = ''   # tu Private API Key de roboflow.com
MATCH_KEY  = '2026-09-06_vs_estudiantes_caseros_h'
VIDEO_PATH = f'/content/drive/MyDrive/analizar-partido/videos/{MATCH_KEY}.mp4'
OUTPUT_DIR = f'/content/drive/MyDrive/analizar-partido/tracking/{MATCH_KEY}'

SAMPLE_FPS    = 5      # frames/seg a procesar
START_SECONDS = 969    # saque inicial confirmado (cartel 00:06 en seg. 975 -> 975-6=969)
TEST_SECONDS  = 900    # F1: primeros 15 min DESDE START_SECONDS; None = hasta el final
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(VIDEO_PATH), f'No existe el video: {VIDEO_PATH}'
assert ROBOFLOW_API_KEY, 'Falta la API key de Roboflow'

# Modelos en Roboflow Universe (el de jugadores incluye la pelota como clase 0)
PLAYER_MODEL_ID = 'football-players-detection-3zvbc/12'
PITCH_MODEL_ID  = 'football-field-detection-f07vi/15'
print('Video OK:', os.path.basename(VIDEO_PATH))
print('Salida  :', OUTPUT_DIR)

## 3. Procesar el video

Mismo pipeline que `src/video/pipeline.py`, autocontenido para Colab. Genera
`positions.csv` con el contrato OPTA 0-100: `frame, t_sec, track_id, role, team, x, y, confidence`.

`t_sec` es relativo a `START_SECONDS` (0 = saque inicial) — así `minute = t_sec // 60`
es directamente el minuto de partido.

In [ ]:
import cv2, json, time
import numpy as np, pandas as pd
import supervision as sv
from inference import get_model
from sports.common.team import TeamClassifier
from sports.common.view import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration

# Clases del modelo de jugadores + dimensiones de cancha (cm) → OPTA 0-100
BALL_ID, GK_ID, PLAYER_ID, REF_ID = 0, 1, 2, 3
ROLE = {PLAYER_ID: 'player', GK_ID: 'goalkeeper', REF_ID: 'referee'}
PITCH_L_CM, PITCH_W_CM = 12000, 7000

player_model = get_model(model_id=PLAYER_MODEL_ID, api_key=ROBOFLOW_API_KEY)
pitch_model  = get_model(model_id=PITCH_MODEL_ID,  api_key=ROBOFLOW_API_KEY)

def infer_dets(frame, conf=0.3):
    return sv.Detections.from_inference(player_model.infer(frame, confidence=conf)[0])

def infer_kpts(frame, conf=0.3):
    return sv.KeyPoints.from_inference(pitch_model.infer(frame, confidence=conf)[0])

# --- 1. Ajustar clasificador de equipos (recortes de jugadores a lo largo del video)
print('Ajustando clasificador de equipos...')
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
crops = []
for fidx in range(0, total_frames, int(fps * 20)):
    if len(crops) > 450: break
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ok, frame = cap.read()
    if not ok: continue
    d = infer_dets(frame)
    crops += [sv.crop_image(frame, xyxy) for xyxy in d[d.class_id == PLAYER_ID].xyxy]
cap.release()
team_clf = TeamClassifier(device='cuda'); team_clf.fit(crops)
print(f'  {len(crops)} recortes usados')

# --- 2. Loop principal
pitch_cfg = SoccerPitchConfiguration()
pitch_vertices = np.array(pitch_cfg.vertices)
tracker = sv.ByteTrack()

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

start_frame = int(START_SECONDS * fps)
end_frame = total_frames
if TEST_SECONDS:
    end_frame = min(end_frame, start_frame + int(TEST_SECONDS * fps))
stride = max(1, round(fps / SAMPLE_FPS))

pos_rows, ball_rows = [], []
t0 = time.time()
for n, fidx in enumerate(range(start_frame, end_frame, stride)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ok, frame = cap.read()
    if not ok: break
    t_sec = (fidx / fps) - START_SECONDS

    det = infer_dets(frame)
    people = det[det.class_id != BALL_ID].with_nms(threshold=0.5)
    people = tracker.update_with_detections(people)
    ball = det[det.class_id == BALL_ID]

    kp = infer_kpts(frame)
    mask = kp.confidence[0] > 0.5
    if mask.sum() < 4:
        continue
    transformer = ViewTransformer(
        source=kp.xy[0][mask].astype(np.float32),
        target=pitch_vertices[mask].astype(np.float32))

    is_player = people.class_id == PLAYER_ID
    teams = np.full(len(people), -1)
    if is_player.sum() > 0:
        pc = [sv.crop_image(frame, xyxy) for xyxy in people.xyxy[is_player]]
        teams[is_player] = team_clf.predict(pc)

    anchors = people.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    xy = transformer.transform_points(anchors.astype(np.float32))
    for i in range(len(people)):
        pos_rows.append({
            'frame': fidx, 't_sec': round(t_sec, 2),
            'track_id': int(people.tracker_id[i]),
            'role': ROLE.get(int(people.class_id[i]), 'player'),
            'team': int(teams[i]),
            'x': round(float(xy[i][0]) / PITCH_L_CM * 100, 2),
            'y': round(float(xy[i][1]) / PITCH_W_CM * 100, 2),
            'confidence': round(float(people.confidence[i]), 3)})

    if len(ball) > 0:
        b = transformer.transform_points(
            ball.get_anchors_coordinates(sv.Position.BOTTOM_CENTER).astype(np.float32))
        ball_rows.append({'frame': fidx, 't_sec': round(t_sec, 2),
                          'x': round(float(b[0][0]) / PITCH_L_CM * 100, 2),
                          'y': round(float(b[0][1]) / PITCH_W_CM * 100, 2),
                          'confidence': round(float(ball.confidence[0]), 3)})

    if (n + 1) % 100 == 0:
        print(f'  {n+1} frames | {t_sec/60:.1f} min de partido | {time.time()-t0:.0f}s transcurridos')
cap.release()

df_pos = pd.DataFrame(pos_rows)
df_ball = pd.DataFrame(ball_rows)
print(f'\nListo: {len(df_pos)} posiciones, {len(df_ball)} de pelota en {time.time()-t0:.0f}s')

## 4. Guardar en Drive

Guarda `positions.csv`, `ball_positions.csv` y `tracking_meta.json` en `OUTPUT_DIR`.
Descargá esos archivos y copialos a tu PC en
`data/temporadas/{año}/partidos/{match_key}/tracking/`.

In [ ]:
from datetime import datetime

# Validación mínima del contrato OPTA antes de guardar
assert not df_pos.empty, 'No se generaron posiciones — revisá el video / la API key'
inside = ((df_pos['x'].between(0, 100)) & (df_pos['y'].between(0, 100))).mean()
print(f'Plausibilidad homografía: {inside*100:.1f}% de puntos dentro de la cancha')

df_pos.to_csv(os.path.join(OUTPUT_DIR, 'positions.csv'), index=False)
if not df_ball.empty:
    df_ball.to_csv(os.path.join(OUTPUT_DIR, 'ball_positions.csv'), index=False)

meta = {
    'match_key': MATCH_KEY,
    'video': os.path.basename(VIDEO_PATH),
    'video_fps': fps, 'sample_fps': SAMPLE_FPS,
    'start_seconds': START_SECONDS, 'test_seconds': TEST_SECONDS,
    'frames_con_posiciones': int(df_pos['frame'].nunique()),
    'fecha': datetime.now().isoformat(timespec='seconds'),
    'entorno': 'colab',
    'modelos': {'player': PLAYER_MODEL_ID, 'pitch': PITCH_MODEL_ID},
}
with open(os.path.join(OUTPUT_DIR, 'tracking_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('\n✅ Guardado en', OUTPUT_DIR)
print('   positions.csv, ball_positions.csv, tracking_meta.json')
print('\nDescargá esos 3 archivos y copialos en tu PC a:')
print(f'   data/temporadas/<año>/partidos/{MATCH_KEY}/tracking/')

# Vista rápida (un instante) para confirmar que parece una formación
import matplotlib.pyplot as plt
inst = df_pos.iloc[(df_pos['t_sec'] - df_pos['t_sec'].median()).abs().argsort()[:60]]
plt.figure(figsize=(10, 6.5))
COL = {0: '#00d4aa', 1: '#cc3333', -1: '#FFD700'}
for tid, g in inst.groupby('team'):
    plt.scatter(g['x'], g['y'], s=120, c=COL.get(tid, 'white'),
                edgecolors='k', label=f'equipo {tid}' if tid >= 0 else 'árbitro')
plt.xlim(0, 100); plt.ylim(0, 100); plt.gca().set_facecolor('#1a3a1a')
plt.title('Tracking — un instante (coordenadas OPTA)'); plt.legend(); plt.show()